# AI Call Moderator v2 — Real Recordings via Kaggle MCP (OAuth)

Continuation of v1. What changed:

- **No hardcoded scenarios** — call-center recordings are pulled live from Kaggle through its **official MCP server** (`https://www.kaggle.com/mcp`), authenticated with **OAuth 2.0** (no API-key files).
- **Speaker identification** — minimal two-stage logic decides *who is the rep and who is the customer* (explained in Sections 5–6).
- **Renames for clarity** — `SOP` → `STANDARD_PROCEDURE`, `CSAT` → `rating` (the predicted 1–5 customer satisfaction rating), and all variables are self-explanatory (`tokenizer`, `language_model`, `TOKEN_USAGE`, `escalation_reason`, …).

**Pipeline:**

```
Kaggle MCP (OAuth 2.0) ──> download recording(s)
        │
        ▼
Whisper transcription ──> speaker separation
   stereo file: one party per channel (free, physical)
   mono file  : one LLM call splits the transcript by context
        │
        ▼
role identification (keyword scores first, LLM only on ties)
        │
        ▼
turn-by-turn moderation ──> violations / sentiment / escalation ──> end-of-call report (rating, standard-procedure checklist)
```

In [ ]:
# 1. Environment setup — installs only what's missing
import importlib, subprocess, sys

def ensure(*packages):
    for package in packages:
        module_name = package.replace("-", "_")
        try:
            importlib.import_module(module_name)
        except ImportError:
            print(f"installing {package} ...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

ensure("torch", "transformers", "accelerate", "mcp", "httpx", "librosa", "soundfile")

import torch, transformers
print(f"torch {torch.__version__} | transformers {transformers.__version__}")
print("GPU available:", torch.cuda.is_available())          # True on ROCm too (HIP backend)

## 2. Connect to Kaggle via MCP with OAuth 2.0

Kaggle's official MCP server supports OAuth 2.0 directly through the MCP protocol
(dynamic client registration → browser consent → authorization code → access token).
Nothing is stored on disk; tokens live only in this notebook session.

Because this lab is headless, we use the standard **manual paste flow**: the notebook prints
the authorization URL, you approve it in your own browser, the browser then redirects to a
`localhost` URL that fails to load (expected — nothing is listening there), and you paste that
full URL back here so the notebook can extract the authorization `code`.

In [ ]:
from urllib.parse import urlparse, parse_qs
from mcp.client.auth import OAuthClientProvider, TokenStorage
from mcp.shared.auth import OAuthClientMetadata

KAGGLE_MCP_URL = "https://www.kaggle.com/mcp"

class InMemoryTokenStorage(TokenStorage):
    """Keeps OAuth tokens in RAM for this session only (no files written)."""
    def __init__(self):
        self.tokens, self.client_info = None, None
    async def get_tokens(self):            return self.tokens
    async def set_tokens(self, tokens):    self.tokens = tokens
    async def get_client_info(self):       return self.client_info
    async def set_client_info(self, info): self.client_info = info

async def show_authorization_url(authorization_url: str):
    print("1) Open this URL in your browser and approve access:\n")
    print(authorization_url)

async def wait_for_pasted_callback():
    pasted_url = input("\n2) After approving, your browser lands on a localhost URL that fails "
                       "to load (expected). Paste that FULL URL here: ").strip()
    query = parse_qs(urlparse(pasted_url).query)
    return query["code"][0], query.get("state", [None])[0]

token_storage = InMemoryTokenStorage()

kaggle_oauth = OAuthClientProvider(
    server_url=KAGGLE_MCP_URL,
    client_metadata=OAuthClientMetadata(
        client_name="call-moderator-v2-notebook",
        redirect_uris=["http://localhost:8765/callback"],
        grant_types=["authorization_code", "refresh_token"],
        response_types=["code"],
        token_endpoint_auth_method="none",   # REQUIRED by Kaggle: public PKCE client
    ),
    storage=token_storage,
    redirect_handler=show_authorization_url,
    callback_handler=wait_for_pasted_callback,
)
print("OAuth provider ready — the consent flow starts on first connect (next cell).")

In [ ]:
# Jupyter supports top-level await, so the async MCP client runs directly in cells.
from contextlib import AsyncExitStack
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

connection_stack = AsyncExitStack()
read_stream, write_stream, _ = await connection_stack.enter_async_context(
    streamablehttp_client(KAGGLE_MCP_URL, auth=kaggle_oauth)
)
kaggle_session = await connection_stack.enter_async_context(ClientSession(read_stream, write_stream))
await kaggle_session.initialize()

KAGGLE_TOOLS = (await kaggle_session.list_tools()).tools
print(f"Connected — {len(KAGGLE_TOOLS)} Kaggle tools available:\n")
for tool in KAGGLE_TOOLS:
    first_description_line = (tool.description or "").splitlines()[0][:70] if tool.description else ""
    print(f"  {tool.name:<42} {first_description_line}")

## 3. Find and download a call-center recordings dataset

The Kaggle MCP tool names can evolve, so instead of hardcoding them the notebook **discovers**
the right tools by keyword from the live `list_tools()` result, and tries a few plausible
argument spellings (the printed tool list above is the source of truth — adjust
`DATASET_REFERENCE` after looking at the search output).

Candidate datasets (audio): `unidpro/call-center-audio`, `axondata/call-center-speech-dataset`.
Text fallback: `oleksiymaliovanyy/call-center-transcripts-dataset` — the pipeline handles both.

In [ ]:
import json

def result_payload(call_result):
    """Flatten an MCP CallToolResult into Python data (JSON-decoded when possible)."""
    text = "\n".join(c.text for c in call_result.content if getattr(c, "text", None))
    try:
        return json.loads(text)
    except (json.JSONDecodeError, TypeError):
        return text

async def call_kaggle(tool_name: str, argument_variants: list):
    """Call a Kaggle tool, trying several plausible argument spellings in order."""
    last_error = None
    for arguments in argument_variants:
        result = await kaggle_session.call_tool(tool_name, arguments)
        if not result.isError:
            return result_payload(result)
        last_error = result_payload(result)
    raise RuntimeError(f"{tool_name} failed with all argument variants. Last error: {last_error}")

def find_tool(*name_keywords):
    """Pick the first Kaggle tool whose name contains all keywords."""
    for tool in KAGGLE_TOOLS:
        if all(keyword in tool.name.lower() for keyword in name_keywords):
            return tool.name
    return None

search_tool_name   = find_tool("search", "dataset") or find_tool("dataset", "list")
download_tool_name = find_tool("download", "dataset") or find_tool("dataset", "url")
print("search tool  :", search_tool_name)
print("download tool:", download_tool_name)

SEARCH_QUERY = "call center audio english"
search_results = await call_kaggle(search_tool_name, [
    {"query": SEARCH_QUERY}, {"search": SEARCH_QUERY}, {"q": SEARCH_QUERY},
])
preview = search_results if isinstance(search_results, str) else json.dumps(search_results, indent=2)
print(preview[:2000])

In [ ]:
import re, httpx, zipfile, pathlib

DATASET_REFERENCE = "unidpro/call-center-audio"   # owner/slug — change based on the search output above
owner_slug, dataset_slug = DATASET_REFERENCE.split("/")

download_info = await call_kaggle(download_tool_name, [
    {"dataset": DATASET_REFERENCE},
    {"ref": DATASET_REFERENCE},
    {"dataset_ref": DATASET_REFERENCE},
    {"owner_slug": owner_slug, "dataset_slug": dataset_slug},
])

def first_url(payload):
    """The download tool returns a (signed) URL somewhere in its payload — grab it."""
    text = payload if isinstance(payload, str) else json.dumps(payload)
    match = re.search(r"https?://[^\s\"']+", text)
    return match.group(0) if match else None

download_url = first_url(download_info)
print("download URL:", (download_url or "NOT FOUND — inspect download_info below")[:120])

DATA_DIR = pathlib.Path("kaggle_call_data")
DATA_DIR.mkdir(exist_ok=True)
archive_path = DATA_DIR / "dataset_download"

request_headers = {}
stored_tokens = await token_storage.get_tokens()           # reuse the OAuth access token
if stored_tokens:
    request_headers["Authorization"] = f"Bearer {stored_tokens.access_token}"

with httpx.stream("GET", download_url, headers=request_headers,
                  follow_redirects=True, timeout=300) as response:
    response.raise_for_status()
    with open(archive_path, "wb") as output_file:
        for chunk in response.iter_bytes():
            output_file.write(chunk)
print(f"downloaded {archive_path.stat().st_size / 1e6:.1f} MB")

if zipfile.is_zipfile(archive_path):
    with zipfile.ZipFile(archive_path) as archive:
        archive.extractall(DATA_DIR)

AUDIO_EXTENSIONS = {".wav", ".mp3", ".flac", ".m4a", ".ogg"}
audio_files = sorted(p for p in DATA_DIR.rglob("*") if p.suffix.lower() in AUDIO_EXTENSIONS)
transcript_files = sorted(p for p in DATA_DIR.rglob("*") if p.suffix.lower() in {".csv", ".txt"})
print(f"{len(audio_files)} audio files, {len(transcript_files)} transcript files")
for path in (audio_files + transcript_files)[:10]:
    print("  ", path)

## 4. Load the local judge model

Same judge as v1 (`Qwen3-4B-Instruct-2507`): greedy decoding for reproducibility, strict
minified-JSON outputs, and a global `TOKEN_USAGE` meter so efficiency stays measurable.

In [ ]:
import re as regex
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
language_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)
language_model.eval()

TOKEN_USAGE = {"prompt": 0, "output": 0, "calls": 0}

@torch.inference_mode()
def generate_json(system_prompt: str, user_prompt: str, max_new_tokens: int = 96) -> dict:
    """One greedy-decoded LLM call that must return JSON. Tracks token usage."""
    messages = [{"role": "system", "content": system_prompt},
                {"role": "user",   "content": user_prompt}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    model_inputs = tokenizer(prompt, return_tensors="pt").to(language_model.device)
    generated = language_model.generate(**model_inputs, max_new_tokens=max_new_tokens,
                                        do_sample=False, temperature=None, top_p=None, top_k=None,
                                        pad_token_id=tokenizer.eos_token_id)
    new_tokens = generated[0][model_inputs.input_ids.shape[1]:]
    TOKEN_USAGE["prompt"] += int(model_inputs.input_ids.shape[1])
    TOKEN_USAGE["output"] += int(len(new_tokens))
    TOKEN_USAGE["calls"]  += 1
    decoded_text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    json_match = regex.search(r"\{.*\}", decoded_text, regex.S)
    if not json_match:
        return {}
    try:
        return json.loads(json_match.group(0))
    except json.JSONDecodeError:
        return {}

print(f"Loaded {MODEL_ID} -> {language_model.device}")

## 5. Transcription + speaker separation

Two cases, cheapest mechanism first:

1. **Stereo recordings** (the norm in telephony: the recorder writes the agent on one channel
   and the caller on the other). Separation is **physical and free** — transcribe each channel
   independently with Whisper, tag utterances `speaker_0` / `speaker_1`, and interleave them by
   start timestamp. Zero LLM tokens, zero errors from crosstalk.
2. **Mono recordings** — no physical separation exists, so we spend **one** LLM call: the model
   reads the raw transcript and splits it into `rep` / `customer` turns by context
   (greetings, requests, account questions make the roles obvious to a context-aware model).

In [ ]:
import numpy as np, librosa
from transformers import pipeline

speech_recognizer = pipeline("automatic-speech-recognition", model="openai/whisper-small",
                             device=0 if torch.cuda.is_available() else -1,
                             chunk_length_s=30, return_timestamps=True)

def transcribe_samples(samples, sampling_rate):
    """Whisper -> list of {'start': seconds, 'text': ...} utterances."""
    output = speech_recognizer({"raw": samples, "sampling_rate": sampling_rate})
    return [{"start": (chunk["timestamp"][0] or 0.0), "text": chunk["text"].strip()}
            for chunk in output["chunks"] if chunk["text"].strip()]

TRANSCRIPT_SPLIT_PROMPT = (
    "You segment a raw call-center transcript into speaker turns and label each turn "
    "'rep' (the service representative) or 'customer'. Keep the original wording; do not invent text. "
     'Reply with ONLY minified JSON: {"turns":[{"role":"rep","text":"..."},{"role":"customer","text":"..."}]}'
)

def split_mono_transcript(full_text: str, max_characters: int = 3500) -> list:
    """Mono fallback: ONE LLM call splits the transcript into labeled turns by context."""
    response = generate_json(TRANSCRIPT_SPLIT_PROMPT,
                             f"Transcript:\n{full_text[:max_characters]}\nJSON:",
                             max_new_tokens=1024)
    return [{"start": index, "speaker": turn.get("role", "customer"), "text": turn["text"]}
            for index, turn in enumerate(response.get("turns", [])) if turn.get("text")]

def utterances_from_audio(audio_path):
    """Returns (utterances, separation_mode). Stereo -> per-channel; mono -> LLM split."""
    samples, sampling_rate = librosa.load(audio_path, sr=16000, mono=False)
    if samples.ndim == 2 and samples.shape[0] == 2:
        utterances = []
        for channel_index in (0, 1):
            for utterance in transcribe_samples(samples[channel_index], sampling_rate):
                utterances.append({**utterance, "speaker": f"speaker_{channel_index}"})
        return sorted(utterances, key=lambda u: u["start"]), "stereo"
    mono_samples = samples if samples.ndim == 1 else samples.mean(axis=0)
    full_text = " ".join(u["text"] for u in transcribe_samples(mono_samples, sampling_rate))
    return split_mono_transcript(full_text), "mono"

print("transcription + separation ready")

## 6. Role identification — who is the rep, who is the customer?

For stereo calls the channels are separated but anonymous (`speaker_0` / `speaker_1`).
Assigning roles is a two-stage decision, cheapest first:

- **Stage 1 (0 tokens):** count role-marker phrases per speaker. Reps say things like
  *"thank you for calling"*, *"how can I help"*, *"is there anything else"*; customers say
  *"my bill"*, *"I was charged"*, *"I want a refund"*. Each rep marker scores +1, each
  customer marker −1. If the two speakers' scores differ by ≥ 2, the answer is unambiguous.
- **Stage 2 (one tiny LLM call, ~30 output tokens):** only when scores are too close, the
  model reads the first 8 utterances and names the rep.

In practice stage 1 resolves almost every call — reps open with near-scripted greetings —
so the average token cost of role identification is close to zero.

In [ ]:
REP_MARKER_PHRASES = (
    "thank you for calling", "how can i help", "how may i help", "this is",
    "let me check", "let me look", "i can help", "i apologize", "i'm sorry for",
    "is there anything else", "have a great day", "your account shows",
)
CUSTOMER_MARKER_PHRASES = (
    "my bill", "my account", "i was charged", "i'm calling about", "im calling about",
    "i want a refund", "i need help", "cancel my", "not working", "speak to a",
    "my order", "my service",
)

ROLE_TIEBREAK_PROMPT = (
    "Two speakers from a call-center call. Decide which one is the customer-service rep. "
    'Reply with ONLY minified JSON: {"rep":"speaker_0"} or {"rep":"speaker_1"}'
)

def identify_roles(utterances: list) -> dict:
    """Map each anonymous speaker tag to 'rep' or 'customer'. LLM used only on ties."""
    marker_scores = {}
    for utterance in utterances:
        text = utterance["text"].lower()
        score_change = (sum(phrase in text for phrase in REP_MARKER_PHRASES)
                        - sum(phrase in text for phrase in CUSTOMER_MARKER_PHRASES))
        marker_scores[utterance["speaker"]] = marker_scores.get(utterance["speaker"], 0) + score_change

    speakers = sorted(marker_scores)
    if len(speakers) == 1:                       # only one voice found
        return {speakers[0]: "rep"}
    score_margin = abs(marker_scores[speakers[0]] - marker_scores[speakers[1]])
    if score_margin >= 2:                        # stage 1: clear keyword verdict, 0 tokens
        rep_speaker = max(speakers, key=lambda s: marker_scores[s])
    else:                                        # stage 2: tiny LLM tiebreak
        call_opening = "\n".join(f"{u['speaker']}: {u['text']}" for u in utterances[:8])
        answer = generate_json(ROLE_TIEBREAK_PROMPT, f"Call opening:\n{call_opening}\nJSON:",
                               max_new_tokens=24)
        rep_speaker = answer.get("rep", speakers[0])
    return {speaker: ("rep" if speaker == rep_speaker else "customer") for speaker in speakers}

def merge_consecutive_turns(labeled_turns: list) -> list:
    """Join back-to-back utterances from the same role into one turn (fewer LLM calls later)."""
    merged = []
    for role, text in labeled_turns:
        if merged and merged[-1][0] == role:
            merged[-1] = (role, merged[-1][1] + " " + text)
        else:
            merged.append((role, text))
    return merged

# self-test on a synthetic opening (no LLM needed: margin >= 2)
_test_utterances = [
    {"speaker": "speaker_0", "text": "Thank you for calling Acme, this is Maya, how can I help?"},
    {"speaker": "speaker_1", "text": "Hi, I was charged twice on my bill and I want a refund."},
]
assert identify_roles(_test_utterances) == {"speaker_0": "rep", "speaker_1": "customer"}
print("role identification OK")

## 7. Policy, standard procedure & the moderator engine

Same enforcement logic as v1, renamed for clarity: the checklist is now
`STANDARD_PROCEDURE` and the end-of-call satisfaction prediction is `rating` (1–5).
Escalation stays deterministic — code applies the rules, the LLM only judges turns.

In [ ]:
# Violation catalog: code -> (severity, definition)
POLICY = {
    # CRITICAL - escalate immediately
    "C1": ("critical", "Improper sensitive data: asking for or reading out a full SSN, "
                       "full card number, CVV, or account password"),
    "C2": ("critical", "Threats, harassment, hate speech, or targeted abuse at a person"),
    "C3": ("critical", "Unethical conduct: soliciting tips/bribes for favors, off-the-books "
                       "deals, falsifying records, or advising someone to lie"),
    # HIGH
    "C4": ("high",     "Unauthorized commitment: promising refunds over $50, compensation, "
                       "or outcomes outside stated policy"),
    "R1": ("high",     "Rep makes/offers account changes WITHOUT identity verification"),
    "R3": ("high",     "Disclosing internal-only data or another customer's information"),
    # MEDIUM
    "R2": ("medium",   "Rep rudeness: dismissive, sarcastic, blaming, or hostile tone"),
}
SEVERITY_BY_CODE = {code: severity for code, (severity, _) in POLICY.items()}

# Standard procedure checklist (scored at end of call)
STANDARD_PROCEDURE = {
    "S1": "Greeting: rep gives name + company and offers help",
    "S2": "Identity verification before account discussion/changes",
    "S3": "Resolution recap: rep restates what was done / next steps",
    "S4": "Closing: rep asks if anything else is needed and closes politely",
}

COMPANY_POLICY_ALLOWANCES = (
    "Refunds up to $50 may be issued instantly; larger amounts require supervisor approval. "
    "Identity verification = full name + last 4 of account + billing ZIP (NEVER full SSN, "
    "full card number, CVV, or password). Reps may offer one goodwill credit up to $25 per call."
)

# Zero-token keyword pre-screener: hits are passed to the LLM as hints to VERIFY, never auto-flagged.
PRESCREEN_PATTERNS = {
    "C1": [r"\b(full|entire|whole|complete)\b.{0,40}\b(ssn|social security|card number)",
           r"\bcvv\b", r"\bsecurity code\b.{0,20}\bback\b", r"\byour password\b"],
    "C2": [r"\b(idiot|stupid|moron|pathetic|shut up)\b",
           r"\bi('ll| will) (sue|hurt|ruin|find) you\b"],
    "C3": [r"\b(venmo|cash app|tip me|kickback)\b", r"\boff[- ]the[- ]books\b"],
    "C4": [r"(refund|credit|comp)\w*\b.{0,40}\$\s?\d{3,}",
           r"\$\s?\d{3,}.{0,40}\b(refund|credit)", r"\bi (promise|guarantee)\b"],
}

def keyword_prescreen(text: str) -> list:
    lowered = text.lower()
    return [code for code, patterns in PRESCREEN_PATTERNS.items()
            if any(regex.search(pattern, lowered) for pattern in patterns)]

print(f"{len(POLICY)} violation codes, {len(STANDARD_PROCEDURE)} standard-procedure items loaded")

In [ ]:
_policy_lines = "\n".join(f"{code}: {definition} [{severity}]"
                          for code, (severity, definition) in POLICY.items())

MODERATOR_SYSTEM_PROMPT = (
    "You are a strict call-center compliance moderator. Judge ONLY the LATEST turn, using context.\n"
    "Violation codes:\n" + _policy_lines + "\n"
    "Company policy: " + COMPANY_POLICY_ALLOWANCES + "\n"
    "Important: a frustrated/venting customer is NOT a violation unless it becomes threats or "
    "targeted abuse (C2). A rep correctly REFUSING an improper request is NOT a violation. "
    "Mentioning a policy limit is NOT a promise.\n"
    'Reply with ONLY minified JSON: {"sentiment":<customer sentiment -2..2 evident right now>,'
    '"violations":[<codes broken in the LATEST turn only>],"reason":"<=12 words"}'
)

REPORT_SYSTEM_PROMPT = (
    "You are a call-center QA analyst. Given a transcript, reply with ONLY minified JSON: "
    '{"rating":<1-5 predicted customer satisfaction rating>,'
    '"procedure":{"S1":0 or 1,"S2":0 or 1,"S3":0 or 1,"S4":0 or 1},'
    '"summary":"<=25 words","coaching":"<=18 words of advice for the rep"}\n'
    "Standard-procedure items: " + json.dumps(STANDARD_PROCEDURE)
)

def analyze_turn(previous_turns, speaker_role, turn_text, keyword_hints):
    context = "\n".join(f"{role}: {text}" for role, text in previous_turns[-4:]) or "(call start)"
    hint_note = f"\nRegex hints to verify (may be false alarms): {keyword_hints}" if keyword_hints else ""
    user_prompt = (f'Context:\n{context}\n\nLATEST {speaker_role.upper()} turn: '
                   f'"{turn_text}"{hint_note}\nJSON:')
    return generate_json(MODERATOR_SYSTEM_PROMPT, user_prompt, max_new_tokens=72)


class CallModerator:
    def __init__(self, call_id: str):
        self.call_id = call_id
        self.turns = []                          # [(speaker_role, text)]
        self.violations_log = []                 # [(turn_number, speaker_role, code)]
        self.customer_sentiment_history = []
        self.escalated = False
        self.escalation_reason = None

    def process_turn(self, speaker_role: str, turn_text: str) -> dict:
        analysis = analyze_turn(self.turns, speaker_role, turn_text, keyword_prescreen(turn_text))
        violations = [code for code in analysis.get("violations", []) if code in POLICY]
        try:
            sentiment = max(-2, min(2, int(analysis.get("sentiment", 0))))
        except (TypeError, ValueError):
            sentiment = 0
        self.turns.append((speaker_role, turn_text))
        turn_number = len(self.turns)
        self.violations_log += [(turn_number, speaker_role, code) for code in violations]
        if speaker_role == "customer":
            self.customer_sentiment_history.append(sentiment)
        if not self.escalated:
            severities = [SEVERITY_BY_CODE[code] for _, _, code in self.violations_log]
            recent_sentiment = self.customer_sentiment_history[-2:]
            if "critical" in severities:
                self._escalate(f"critical violation {violations} (rule 1)")
            elif severities.count("high") >= 2:
                self._escalate("two high-severity violations (rule 2)")
            elif len(recent_sentiment) == 2 and all(s <= -2 for s in recent_sentiment):
                self._escalate("sustained very negative customer sentiment (rule 3)")
        return {"turn_number": turn_number, "speaker_role": speaker_role,
                "sentiment": sentiment, "violations": violations,
                "reason": analysis.get("reason", ""), "escalated": self.escalated}

    def _escalate(self, reason: str):
        self.escalated, self.escalation_reason = True, reason

    def end_of_call_report(self) -> dict:
        transcript = "\n".join(f"{role}: {text}" for role, text in self.turns)
        report = generate_json(REPORT_SYSTEM_PROMPT, f"Transcript:\n{transcript}\nJSON:",
                               max_new_tokens=128)
        report.setdefault("rating", None)
        report.setdefault("procedure", {})
        return report

print("moderator engine ready")

## 8. Moderate the pulled recordings

End-to-end on real data: download → transcribe → separate speakers → identify roles →
moderate turn-by-turn → end-of-call report. If the chosen dataset turned out to be
transcripts (CSV) instead of audio, the text fallback path is used automatically.
`MAX_CALLS_TO_MODERATE` / `MAX_TURNS_PER_CALL` cap the token budget.

In [ ]:
MAX_CALLS_TO_MODERATE = 2
MAX_TURNS_PER_CALL    = 24

def moderate_call(call_id: str, turns: list):
    moderator = CallModerator(call_id)
    escalation_announced = False
    print("=" * 78); print(f"CALL {call_id}"); print("=" * 78)
    for speaker_role, turn_text in turns[:MAX_TURNS_PER_CALL]:
        result = moderator.process_turn(speaker_role, turn_text)
        sentiment_note = (f" (sentiment {result['sentiment']:+d})"
                          if speaker_role == "customer" else "")
        violation_note = (f"   <- VIOLATION {result['violations']}: {result['reason']}"
                          if result["violations"] else "")
        print(f"[{result['turn_number']:02d}] {speaker_role:>8}{sentiment_note}: "
              f"{turn_text[:160]}{violation_note}")
        if moderator.escalated and not escalation_announced:
            escalation_announced = True
            print(f"     *** ESCALATED -> {moderator.escalation_reason} ***")
    report = moderator.end_of_call_report()
    procedure_marks = ", ".join(f"{item}={mark}" for item, mark in report.get("procedure", {}).items())
    print(f"\n  end-of-call | rating: {report.get('rating')}/5 | standard procedure: {procedure_marks}")
    print(f"  summary : {report.get('summary', '')}")
    print(f"  coaching: {report.get('coaching', '')}\n")
    return moderator, report

moderation_results = {}

if audio_files:
    for audio_path in audio_files[:MAX_CALLS_TO_MODERATE]:
        utterances, separation_mode = utterances_from_audio(audio_path)
        print(f"\n{audio_path.name}: {len(utterances)} utterances ({separation_mode} separation)")
        if separation_mode == "stereo":
            speaker_roles = identify_roles(utterances)
            print("  roles:", speaker_roles)
            labeled_turns = [(speaker_roles[u["speaker"]], u["text"]) for u in utterances]
        else:
            labeled_turns = [(u["speaker"], u["text"]) for u in utterances]   # mono path is pre-labeled
        turns = merge_consecutive_turns(labeled_turns)
        moderation_results[audio_path.name] = moderate_call(audio_path.name, turns)
elif transcript_files:
    ensure("pandas"); import pandas as pd
    table = pd.read_csv(transcript_files[0])
    text_column = max(table.columns, key=lambda c: table[c].astype(str).str.len().mean())
    print(f"text fallback: using column '{text_column}' of {transcript_files[0].name}")
    for row_index, raw_transcript in enumerate(table[text_column].head(MAX_CALLS_TO_MODERATE)):
        utterances = split_mono_transcript(str(raw_transcript))
        turns = merge_consecutive_turns([(u["speaker"], u["text"]) for u in utterances])
        moderation_results[f"transcript_{row_index}"] = moderate_call(f"transcript_{row_index}", turns)
else:
    print("No audio or transcript files found — check the download cell output.")

## 9. Efficiency report

Real recordings have no ground-truth labels, so v2 reports cost rather than accuracy
(v1's labeled scenarios remain the accuracy benchmark).

In [ ]:
calls_made = max(TOKEN_USAGE["calls"], 1)
print(f"Calls moderated      : {len(moderation_results)}")
print(f"LLM calls            : {TOKEN_USAGE['calls']}")
print(f"Token usage          : {TOKEN_USAGE['prompt']:,} prompt + {TOKEN_USAGE['output']:,} output")
print(f"Avg per LLM call     : {TOKEN_USAGE['prompt'] // calls_made} prompt / "
      f"{TOKEN_USAGE['output'] // calls_made} output tokens")
for call_id, (moderator, report) in moderation_results.items():
    detected_codes = sorted({code for _, _, code in moderator.violations_log})
    print(f"  {call_id:<30} escalated={moderator.escalated} "
          f"violations={detected_codes or '-'} rating={report.get('rating')}/5")

## Notes & next steps

- The Kaggle MCP tool names/arguments are **discovered at runtime** (Section 3) because the
  server's schema can evolve — the printed `list_tools()` output is always the source of truth.
- Stereo channel separation is exact; for mono calls needing higher fidelity than the
  one-shot LLM split, add `pyannote.audio` diarization and keep the same role-identification stage.
- All enforcement still lives in `POLICY`, `STANDARD_PROCEDURE`, `COMPANY_POLICY_ALLOWANCES`,
  and the escalation rules inside `CallModerator.process_turn` — edit and re-run.